In [1]:
"""
OXE per-subset diversity scatter  ->  paper Figure 1
====================================================
Streams natural-language instructions from OXE subsets (HF mirror
jxu124/OpenX-Embodiment), computes T1 verb-entropy + tail stats per subset,
overlays the three in-hand datasets (DROID / AgiBot / RoboMIND), and saves a
scatter showing: entropy is nearly flat while tail structure (Gini, breadth)
varies enormously -> entropy-only diversity metrics are insufficient.

Requires:  pip install datasets
Note: streaming pulls tar shards on the fly (~0.7-1.5 GB bandwidth per subset,
not permanently stored). Start with a few subsets; edit OXE_LANG_SUBSETS / CAP.
Env: Python 3.12, PyTorch-only, Windows (I:/ROBO).
"""

import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = r"I:/ROBO"
OUT = os.path.join(BASE, "outputs")
os.makedirs(OUT, exist_ok=True)

# OXE subsets that carry language instructions (edit / extend freely)
OXE_LANG_SUBSETS = [
    "taco_play", "jaco_play", "berkeley_autolab_ur5", "viola", "bc_z",
    "austin_buds_dataset_converted_externally_to_rlds",
    # heavier but canonical (uncomment to add): "fractal20220817_data", "bridge",
]
CAP = 120                       # episodes streamed per subset (enough for the distribution)
SKIP_KEYS = ("image", "rgb", "depth", "observation", "pixels", "embedding",
             "flow", "point_cloud", "wrist")


# --------------------------- shared helpers ---------------------------
def _first_verb(s):
    t = re.findall(r"[a-zA-Z]+", s.lower())
    return t[0] if t else None


def stats(counter):
    if not counter:
        return None
    vals = np.array(sorted(counter.values(), reverse=True), dtype=float)
    p = vals / vals.sum()
    H = float(-(p * np.log(p)).sum())
    asc = np.sort(vals); cum = np.cumsum(asc); n = len(vals)
    gini = (n + 1 - 2 * np.sum(cum) / cum[-1]) / n
    return {"H": round(H, 3), "n_verbs": n,
            "top2": round(float(p[:2].sum()), 3), "gini": round(float(gini), 3)}


def find_language(x, depth=0):
    """Recursively yield instruction-like strings; skips image/observation keys."""
    if depth > 6:
        return
    if isinstance(x, bytes):
        try:
            x = x.decode("utf-8", "ignore")
        except Exception:
            return
    if isinstance(x, dict):
        for k, v in x.items():
            kl = str(k).lower()
            if any(sk in kl for sk in SKIP_KEYS):
                continue
            if isinstance(v, (str, bytes)) and any(t in kl for t in
                                                   ("language", "instruction", "natural")):
                s = v.decode("utf-8", "ignore") if isinstance(v, bytes) else v
                if s and s.strip():
                    yield s.strip()
            else:
                yield from find_language(v, depth + 1)
    elif isinstance(x, (list, tuple)):
        for v in x[:60]:
            yield from find_language(v, depth + 1)


def code_subset(name, cap=CAP):
    from datasets import load_dataset
    ds = load_dataset("jxu124/OpenX-Embodiment", name, split="train",
                      streaming=True, trust_remote_code=True)
    c = Counter(); seen = 0
    for ex in ds:
        for instr in find_language(ex):
            v = _first_verb(instr)
            if v:
                c[v] += 1
        seen += 1
        if seen >= cap:
            break
    return c, seen


def probe():
    """Test datasets-library compatibility on ONE subset before the full loop."""
    try:
        import datasets  # noqa
    except ImportError:
        print("نصب کنید:  pip install datasets"); return False
    print("probe: streaming 5 episodes of 'taco_play' ...")
    try:
        c, seen = code_subset("taco_play", cap=5)
        print("  OK ->", stats(c), "| verbs found from", seen, "eps:", c.most_common(5))
        return True
    except Exception as e:
        print("  probe FAILED:", type(e).__name__, "-", str(e)[:120])
        print("  → نسخهٔ datasets شما احتمالاً loading-script را رد می‌کند.")
        print("    همین پیام خطا را بفرست تا نسخهٔ fallback (دانلود مستقیم shard) را بدهم.")
        return False


def main():
    if not probe():
        return
    rows = []
    for name in OXE_LANG_SUBSETS:
        try:
            c, seen = code_subset(name); st = stats(c)
            if st:
                rows.append({"subset": name[:26], "source": "OXE", **st, "episodes": seen})
                print(f"{name[:30]:30s} -> {st}  ({seen} eps)")
            else:
                print(f"{name[:30]:30s} -> no language found, skip")
        except Exception as e:
            print(f"{name[:30]:30s} -> skip ({type(e).__name__})")

    inhand = [
        {"subset": "DROID",    "source": "main", "H": 2.716, "n_verbs": 299, "top2": 0.380, "gini": 0.956, "episodes": 50092},
        {"subset": "AgiBot",   "source": "main", "H": 2.726, "n_verbs": 34,  "top2": 0.404, "gini": 0.635, "episodes": 34512},
        {"subset": "RoboMIND", "source": "main", "H": 2.616, "n_verbs": 50,  "top2": 0.509, "gini": 0.748, "episodes": 479},
    ]
    df = pd.DataFrame(rows + inhand)
    df.to_csv(os.path.join(OUT, "figure1_entropy_vs_tail.csv"), index=False)

    fig, ax = plt.subplots(figsize=(7, 5))
    for src, mk, col in [("OXE", "o", "#4C78A8"), ("main", "*", "#E45756")]:
        sub = df[df.source == src]
        if len(sub):
            ax.scatter(sub.H, sub.gini, s=40 + 4 * np.sqrt(sub.n_verbs),
                       marker=mk, c=col, alpha=0.75, label=src,
                       edgecolor="k", linewidth=0.4)
    for _, r in df[df.source == "main"].iterrows():
        ax.annotate(r["subset"], (r.H, r.gini), fontsize=8,
                    xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("verb entropy (nats) — nearly invariant")
    ax.set_ylabel("verb Gini — highly variable")
    ax.set_title("Entropy hides tail structure across robot datasets")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, "figure1_entropy_vs_tail.png"), dpi=150)
    print("\nsaved -> figure1_entropy_vs_tail.csv + .png")
    print(df.to_string(index=False))


if __name__ == "__main__":
    main()

c:\Users\PC369\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'jxu124/OpenX-Embodiment' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


probe: streaming 5 episodes of 'taco_play' ...


c:\Users\PC369\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC369\.cache\huggingface\hub\datasets--jxu124--OpenX-Embodiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


  probe FAILED: RuntimeError - Dataset scripts are no longer supported, but found OpenX-Embodiment.py
  → نسخهٔ datasets شما احتمالاً loading-script را رد می‌کند.
    همین پیام خطا را بفرست تا نسخهٔ fallback (دانلود مستقیم shard) را بدهم.


Oxe lerobot fallback · PY

In [2]:
"""
OXE via LeRobot mirror (script-free fallback for Figure 1)
=========================================================
The `datasets` library dropped loading-script support, so jxu124/OpenX-Embodiment
no longer loads. LeRobot-format OXE mirrors are plain parquet + tiny metadata.
We download ONLY the task metadata (natural-language task strings, a few KB per
subset) and compute per-subset verb stats. No TensorFlow, no video.

NOTE ON GRANULARITY: this uses UNIQUE task strings (from meta/tasks). For the
final figure we will align granularity with the 3 main datasets (per-episode
frequency via meta/episodes). This first pass is for coverage/breadth, not final
frequency-weighted entropy.

Env: Python 3.12, PyTorch-only, Windows (I:/ROBO). Needs a valid HF login.
"""

import os
import re
import json
from collections import Counter

import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

BASE = r"I:/ROBO"
OUT = os.path.join(BASE, "outputs")
META = os.path.join(BASE, "data/oxe_lerobot_meta")
os.makedirs(OUT, exist_ok=True); os.makedirs(META, exist_ok=True)
api = HfApi()
AUTHOR = "IPEC-COMMUNITY"
LIMIT = 25            # how many OXE subsets to process (raise later)


def _fv(s):
    t = re.findall(r"[a-z]+", s.lower())
    return t[0] if t else None


def stats(counter):
    if not counter:
        return None
    vals = np.array(sorted(counter.values(), reverse=True), dtype=float)
    p = vals / vals.sum()
    H = float(-(p * np.log(p)).sum())
    asc = np.sort(vals); cum = np.cumsum(asc); n = len(vals)
    gini = (n + 1 - 2 * np.sum(cum) / cum[-1]) / n
    return {"H": round(H, 3), "n_verbs": n,
            "top2": round(float(p[:2].sum()), 3), "gini": round(float(gini), 3)}


def list_oxe_repos():
    ds = list(api.list_datasets(author=AUTHOR, limit=500))
    repos = [d.id for d in ds if "lerobot" in d.id.lower()]
    print(f"found {len(repos)} lerobot repos under {AUTHOR}")
    return repos[:LIMIT]


def tasks_of(repo):
    files = api.list_repo_files(repo, repo_type="dataset")
    cand = [f for f in files if f.startswith("meta/") and "task" in f.lower()]
    if not cand:
        return []
    lp = hf_hub_download(repo, repo_type="dataset", filename=cand[0],
                         local_dir=os.path.join(META, repo.replace("/", "_")))
    out = []
    if lp.endswith(".jsonl"):
        for line in open(lp, encoding="utf-8"):
            line = line.strip()
            if line:
                try:
                    out.append(json.loads(line).get("task", ""))
                except Exception:
                    pass
    elif lp.endswith(".parquet"):
        df = pd.read_parquet(lp)
        col = next((c for c in df.columns if "task" in c.lower() or "instr" in c.lower()),
                   df.columns[0])
        out = df[col].dropna().astype(str).tolist()
    return [t for t in out if t]


def main():
    repos = list_oxe_repos()
    rows = []
    for repo in repos:
        try:
            tasks = tasks_of(repo)
            if not tasks:
                print(f"{repo.split('/')[-1][:40]:40s} -> no task meta"); continue
            c = Counter(v for v in (_fv(t) for t in tasks) if v)
            st = stats(c)
            if st:
                rows.append({"subset": repo.split("/")[-1][:30], "source": "OXE",
                             **st, "n_tasks": len(tasks)})
                print(f"{repo.split('/')[-1][:40]:40s} -> {st} ({len(tasks)} tasks)")
        except Exception as e:
            print(f"{repo.split('/')[-1][:40]:40s} -> skip ({type(e).__name__})")
    pd.DataFrame(rows).to_csv(os.path.join(OUT, "oxe_lerobot_metrics.csv"), index=False)
    print("\nsaved -> oxe_lerobot_metrics.csv")


if __name__ == "__main__":
    main()

found 37 lerobot repos under IPEC-COMMUNITY
bridge_orig_lerobot                      -> {'H': 2.396, 'n_verbs': 408, 'top2': 0.576, 'gini': 0.955} (19973 tasks)
fractal20220817_data_lerobot             -> {'H': 1.096, 'n_verbs': 6, 'top2': 0.816, 'gini': 0.608} (598 tasks)
droid_lerobot                            -> {'H': 2.747, 'n_verbs': 216, 'top2': 0.368, 'gini': 0.935} (31307 tasks)
nyu_franka_play_dataset_lerobot          -> {'H': -0.0, 'n_verbs': 1, 'top2': 1.0, 'gini': 0.0} (1 tasks)
dlr_edan_shared_control_lerobot          -> {'H': -0.0, 'n_verbs': 1, 'top2': 1.0, 'gini': 0.0} (9 tasks)
nyu_door_opening_surprising_effectivenes -> {'H': -0.0, 'n_verbs': 1, 'top2': 1.0, 'gini': 0.0} (1 tasks)
viola_lerobot                            -> {'H': 1.099, 'n_verbs': 3, 'top2': 0.667, 'gini': 0.0} (3 tasks)
jaco_play_lerobot                        -> {'H': 0.398, 'n_verbs': 2, 'top2': 1.0, 'gini': 0.364} (88 tasks)
roboturk_lerobot                         -> {'H': 1.099, 'n_verbs': 3, '